# Paper 2 · Langkah 1: Unduh dan ambil sampel AI-ArtBench

Notebook ini mengunduh dataset AI-ArtBench dari Kaggle, mendata isinya, mengambil sampel 3.000 gambar secara acak dengan seed tetap, lalu menyalin **file aslinya tanpa diubah** ke Google Drive beserta spreadsheet induk (manifest).

Jalankan sel satu per satu dari atas. Kalau ada error, salin pesannya ke Claude.

Sumber: Silva et al. (2024), *ArtBrain*, arXiv:2412.01512. Dataset: kaggle.com/datasets/ravidussilva/real-ai-art

## 1. Hubungkan Google Drive dan siapkan folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/Paper2'
RAW_OUT = f'{PROJECT}/data_asli/ai_artbench_sample'   # file asli, jangan pernah diubah
META_OUT = f'{PROJECT}/catatan'
for p in [RAW_OUT, META_OUT, f'{PROJECT}/data_olahan', f'{PROJECT}/hasil']:
    os.makedirs(p, exist_ok=True)
print('Folder proyek siap di', PROJECT)

## 2. Unduh dataset dari Kaggle

Sebelum menjalankan sel ini: login ke kaggle.com → Settings → tab API Tokens → **Create Legacy API Key**. File `kaggle.json` akan terunduh. Unggah file itu saat diminta di bawah.

Dataset diunduh ke disk sementara Colab (`/content/aiartbench`), bukan ke Drive, karena ukurannya besar.

In [ ]:
!pip -q install kaggle
from google.colab import files
uploaded = files.upload()   # pilih kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d ravidussilva/real-ai-art -p /content/aiartbench --unzip -q
!du -sh /content/aiartbench

## 3. Periksa struktur folder

Sel ini **tidak mengubah apa pun**. Ia hanya menampilkan nama-nama folder dan jumlah gambar di tiap folder, supaya kita bisa memastikan cara membaca label sumber (manusia, Latent Diffusion, Stable Diffusion) dan gaya sudah benar. Salin tabel hasilnya ke Claude kalau ada yang terlihat aneh.

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path('/content/aiartbench')
EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
rows = []
for p in ROOT.rglob('*'):
    if p.is_file() and p.suffix.lower() in EXT:
        rel = p.relative_to(ROOT).parts
        rows.append({'path': str(p), 'folder': p.parent.name, 'parts': '/'.join(rel[:-1])})
inv = pd.DataFrame(rows)
print('Total gambar ditemukan:', len(inv))
summary = inv.groupby('parts').size().rename('jumlah').reset_index()
pd.set_option('display.max_rows', 200)
summary

## 4. Baca label sumber, gaya, dan split dari nama folder

Aturannya: nama folder yang mengandung `AI_LD` berarti Latent Diffusion, `AI_SD` berarti Stable Diffusion, selain itu karya manusia. Sisa nama folder adalah gaya. Split (train/test) diambil dari nama folder di atasnya.

Sel ini berhenti dengan pesan kalau hasilnya tidak sesuai harapan (3 sumber × 10 gaya). Kalau berhenti, kirim hasil sel 3 ke Claude.

In [ ]:
import re

def parse(row):
    f = row['folder']
    fl = f.lower()
    if 'ai_ld' in fl:
        source = 'latent_diffusion'
    elif 'ai_sd' in fl:
        source = 'stable_diffusion'
    else:
        source = 'human'
    style = re.sub(r'(?i)^ai_(ld|sd)_', '', f).strip().lower().replace('-', '_')   # ukiyo-e -> ukiyo_e
    parts = row['parts'].lower().split('/')
    split = 'train' if 'train' in parts else ('test' if 'test' in parts else 'unknown')
    return pd.Series({'source': source, 'style': style, 'split': split})

inv = pd.concat([inv, inv.apply(parse, axis=1)], axis=1)
inv['label'] = (inv['source'] != 'human').map({True: 'ai', False: 'human'})

table = inv.pivot_table(index='style', columns=['split', 'source'], values='path', aggfunc='count', fill_value=0)
display(table)

n_sources, n_styles = inv['source'].nunique(), inv['style'].nunique()
assert n_sources == 3 and n_styles == 10, (
    f'Ditemukan {n_sources} sumber dan {n_styles} gaya, bukan 3 dan 10. '
    'Kirim hasil sel 3 dan sel ini ke Claude sebelum lanjut.')
print('OK: 3 sumber dan 10 gaya terbaca dengan benar.')

## 5. Ambil sampel 3.000 gambar

Per gaya: 150 karya manusia, 75 Latent Diffusion, 75 Stable Diffusion. Hanya dari split **train**, supaya split test resmi tetap utuh sebagai cadangan uji tambahan. Seed tetap (42), jadi hasil sampel selalu sama kalau dijalankan ulang.

In [ ]:
SEED = 42
PLAN = {'human': 150, 'latent_diffusion': 75, 'stable_diffusion': 75}

pool = inv[inv['split'] == 'train']
picked = []
for (style, source), g in pool.groupby(['style', 'source']):
    n = PLAN[source]
    assert len(g) >= n, f'Gambar {source}/{style} kurang dari {n}'
    picked.append(g.sample(n=n, random_state=SEED))
sample = pd.concat(picked).sort_values(['style', 'source', 'path']).reset_index(drop=True)
sample.insert(0, 'sample_id', [f'AB{str(i).zfill(4)}' for i in range(1, len(sample) + 1)])
print('Jumlah sampel:', len(sample))
display(sample.groupby(['label', 'source']).size())

## 6. Salin file asli ke Drive dan buat manifest

File disalin byte demi byte (tidak dibuka ulang, tidak di-resize, tidak disimpan ulang), supaya format dan kompresi aslinya utuh untuk RQ1. Manifest mencatat format, ukuran file, dimensi, mode warna, dan hash SHA-256 setiap gambar.

In [ ]:
import shutil, hashlib
from PIL import Image

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

records = []
for r in sample.itertuples():
    src = Path(r.path)
    dst_dir = Path(RAW_OUT) / r.source / r.style
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / f'{r.sample_id}{src.suffix.lower()}'
    shutil.copy2(src, dst)
    try:
        with Image.open(dst) as im:
            fmt, (w, h), mode = im.format, im.size, im.mode
        ok = True
    except Exception as e:
        fmt, w, h, mode, ok = None, None, None, None, False
    records.append({
        'sample_id': r.sample_id, 'label': r.label, 'source': r.source, 'style': r.style,
        'split_asal': r.split, 'file_asal': str(src.relative_to(ROOT)), 'file_drive': str(dst),
        'format': fmt, 'ekstensi': src.suffix.lower(), 'width': w, 'height': h, 'mode': mode,
        'file_size_bytes': dst.stat().st_size, 'sha256': sha256(dst), 'bisa_dibuka': ok,
    })

manifest = pd.DataFrame(records)
manifest.to_csv(f'{META_OUT}/manifest_ai_artbench_sample.csv', index=False)
inv.drop(columns=['parts']).to_csv(f'{META_OUT}/inventaris_ai_artbench_lengkap.csv', index=False)
print('Manifest tersimpan:', f'{META_OUT}/manifest_ai_artbench_sample.csv')

## 7. Cek kesehatan data dan gambaran awal RQ1

Mengecek file rusak dan duplikat, lalu meringkas format dan dimensi per sumber. Ringkasan terakhir adalah gambaran awal apakah ada perbedaan teknis antar-kelas yang bisa dipakai model sebagai jalan pintas.

In [ ]:
print('File yang tidak bisa dibuka:', (~manifest['bisa_dibuka']).sum())
dup = manifest[manifest.duplicated('sha256', keep=False)]
print('File duplikat (hash sama):', len(dup))
if len(dup):
    display(dup[['sample_id', 'source', 'style', 'sha256']])

print('\nFormat per sumber:')
display(manifest.groupby(['source', 'format']).size().unstack(fill_value=0))
print('\nDimensi per sumber:')
display(manifest.groupby('source')[['width', 'height']].describe().round(0))
print('\nUkuran file (KB) per sumber:')
display((manifest['file_size_bytes'] / 1024).groupby(manifest['source']).describe().round(1))

## Selesai

Yang tersimpan di Drive (`MyDrive/Paper2`):
- `data_asli/ai_artbench_sample/` — 3.000 file asli, dikelompokkan per sumber dan gaya
- `catatan/manifest_ai_artbench_sample.csv` — spreadsheet induk sampel
- `catatan/inventaris_ai_artbench_lengkap.csv` — daftar seluruh isi dataset

Kirim ke Claude: hasil sel 4, 5, dan 7 (cukup screenshot atau salin teksnya). Catat juga tanggal unduh di log kerja.